# 05 Overfitting and Generalization

In [ ]:
# User-editable papermill/environment configuration
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT = Path(
    os.environ.get(
        "WWGPT_RESULTS_ROOT",
        os.environ.get(
            "RESULTS_ROOT",
            "/tmp/wwpgd_v2/real_level0_results_v5/"
            "experiments/level_00/multiplier_20",
        ),
    )
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))

ANALYSIS_DIR = RESULTS_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_LEGACY = False
EXPECTED_SEEDS = {1337, 2027, 4099, 7919, 104729}


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd().resolve()
repo = cwd if (cwd/'src'/'wwgpt').exists() else cwd.parent
if str(repo/'src') not in sys.path:
    sys.path.insert(0, str(repo/'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from wwgpt.analysis import *
RESULTS_ROOT = resolve_experiment_root(RESULTS_ROOT)
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run log: {RUN_LOG}')
print(f'PID file: {PID_FILE}')
from wwgpt.publication_plots import PublicationPlotConfig, plot_metric, build_all_publication_figures


Generalization and fixed-probe metrics for canonical schema-v2 pairs.

In [ ]:
candidates = discover_pair_candidates(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
canonical_pairs, pair_audit = select_canonical_pairs(candidates)
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
inventory = build_run_inventory(runs)
pair_audit.to_csv(ANALYSIS_DIR/'pair_candidate_audit.csv', index=False)
inventory.to_csv(ANALYSIS_DIR/'run_inventory.csv', index=False)
print(f'Canonical pairs: {len(canonical_pairs)}')
display(pair_audit)
display(inventory)


In [ ]:
# Publication plots are generated through the canonical API.
# The API exports a vector PDF, a >=300 DPI PNG, exact source data, and metadata for every figure.
plot_config = PublicationPlotConfig(band='mean_std', dpi=300)
def notebook_plot_metric(metric, filename, ylabel=None, final_frac=None):
    # final_frac is retained for backwards-compatible notebook execution, but canonical
    # publication figures do not interpolate/smooth across incompatible token budgets.
    figure_name = Path(filename).stem
    return plot_metric(runs, metric, ANALYSIS_DIR, figure_name=figure_name, ylabel=ylabel, config=plot_config)
